In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.append(f'{os.getcwd()}/irc_gym')
from ult import *
from irc_gym.irc.model import FuncBeliefModel
from stable_baselines3 import PPO
from auditoryforage.AF_env import AuditoryForaging as AFtask


In [3]:
modelname = 'test'

epoch_size = 333
n_epoch = 3
n_seed = 1
seed = 0
np.random.seed(seed)

# vary conditions
vmin, vmax = 0.1, 5
att_cost_list = np.linspace(vmin, vmax, 5)
plist = [np.array([.5, .7])]
node_list = [25]
# fixed params
# attcost = -1.6
facost = -50
env_param = [0, None, 0, .25, facost, 0, 0]

# for plotting
no_episodes = 2222  # for eval plot during training
plot_no_episodes = 555 # for raster plot
minlen, maxlen=0,np.inf
nbin=15
bins=np.linspace(0,1,nbin)


In [4]:
# helper and plot function
def process_one_subdf_df(subdf,no_signal_nodes=25, no_penalty_nodes=1):
    '''process the subdf data into result lists. the input is a df'''
    hit_count = 0
    miss_count = 0
    false_alarm_count = 0
    noise_time_before_lick = 0
    signal_time_before_lick = 0
    total_noise_time = 0
    total_signal_time = 0
    total_reward = 0

    attention_time_points_across_subdfs = []
    subdf_length_across_subdfs = []
    hit_reaction_time_across_subdfs = []
    fa_reaction_time_across_subdfs = []

    # process single ep data
    if subdf['actions'][-1][0] == 1:
        if subdf['states'][-1][0] >  no_signal_nodes +  no_penalty_nodes:
            hit_count += 1
            signal_time_before_lick_curr_subdf = count_no_elements(
                subdf['states'], 1,  no_signal_nodes)
            hit_reaction_time_across_subdfs.append(
                signal_time_before_lick_curr_subdf)
            signal_time_before_lick += signal_time_before_lick_curr_subdf
        else:
            false_alarm_count += 1
            noise_time_before_lick_curr_subdf = count_no_elements(
                subdf['states'], 0, 0)
            fa_reaction_time_across_subdfs.append(
                noise_time_before_lick_curr_subdf)
            noise_time_before_lick += noise_time_before_lick_curr_subdf
    else:
        miss_count += 1
    total_signal_time += count_no_elements(
        subdf['states'], 1,  no_signal_nodes)
    total_noise_time += count_no_elements(subdf['states'], 0, 0)
    total_reward += sum(subdf['rewards'])
    # attention_time_points_across_subdfs.append(np.where(subdf['actions'] % env.no_attention_modes > 0)[0])
    attention_time_points_across_subdfs.append(
        np.where(np.array([elt[1] for elt in subdf['actions']]) == 1)[0])
    subdf_length_across_subdfs.append(len(subdf['states']))
    # end process single ep data
    food_reward_idx = subdf['att_idx']
    return (food_reward_idx,
            hit_count,
            miss_count,
            false_alarm_count,
            noise_time_before_lick,
            signal_time_before_lick,
            total_noise_time,
            total_signal_time,
            total_reward,
            attention_time_points_across_subdfs,
            subdf_length_across_subdfs,
            hit_reaction_time_across_subdfs,
            fa_reaction_time_across_subdfs
            )


In [5]:

def hit_reaction_plot(df):
    ''' '''
    # process data
    hit_count_list = [[0] for _ in range(len(att_cost_list))]
    miss_count_list = [[0] for _ in range(len(att_cost_list))]
    false_alarm_list = [[0] for _ in range(len(att_cost_list))]

    noise_time_before_lick_list = [[0] for _ in range(len(att_cost_list))]
    signal_time_before_lick_list = [[0] for _ in range(len(att_cost_list))]
    total_noise_time_list = [[0] for _ in range(len(att_cost_list))]
    total_signal_time_list = [[0] for _ in range(len(att_cost_list))]
    # dist naming space: list of list of num. big list of each reward cond. small list of each trial's summary stats, which is a single number.
    total_reward_dist= [[] for _ in range(len(att_cost_list))]
    rt_hit_dist=[[] for _ in range(len(att_cost_list))]
    rt_fa_dist=[[] for _ in range(len(att_cost_list))]
    total_att_time_dist = [[] for _ in range(len(att_cost_list))]

    for i in range(len(df)):
        (food_reward_idx,
        hit_count,
        miss_count,
        false_alarm_count,

        noise_time_before_lick,
        signal_time_before_lick,
        total_noise_time,
        total_signal_time,

        total_reward,

        attention_time_points_across_episodes,
        episode_length_across_episodes,
        hit_reaction_time_across_episodes,
        fa_reaction_time_across_episodes
        ) = process_one_subdf_df(df.iloc[i])

        hit_count_list[food_reward_idx][0] += (hit_count)
        miss_count_list[food_reward_idx][0] += (miss_count)
        false_alarm_list[food_reward_idx][0] += (false_alarm_count)

        noise_time_before_lick_list[food_reward_idx][0] += (noise_time_before_lick)
        signal_time_before_lick_list[food_reward_idx][0] += (
            signal_time_before_lick)
        total_noise_time_list[food_reward_idx][0] += (total_noise_time)
        total_signal_time_list[food_reward_idx][0] += (total_signal_time)

        total_reward_dist[food_reward_idx].append(total_reward)
        rt_hit_dist[food_reward_idx].append(signal_time_before_lick)
        rt_fa_dist[food_reward_idx].append(noise_time_before_lick)
        total_att_time_dist[food_reward_idx].append(len(attention_time_points_across_episodes[0]))


    # 3 curve plot
    hit_count_list, miss_count_list, false_alarm_list = np.array(
        hit_count_list), np.array(miss_count_list), np.array(false_alarm_list)
    hit_prob = [hit_count_list[i]/(hit_count_list[i]+miss_count_list[i] +
                                    false_alarm_list[i]) for i in range(len(hit_count_list))]
    miss_prob = [miss_count_list[i]/(hit_count_list[i]+miss_count_list[i] +
                                        false_alarm_list[i]) for i in range(len(hit_count_list))]
    fa_prob = [false_alarm_list[i]/(hit_count_list[i]+miss_count_list[i] +
                                    false_alarm_list[i]) for i in range(len(hit_count_list))]

    xs=np.arange(len(att_cost_list))
    fig, ax=plt.subplots(1,1, figsize=(5,4))

    ax.plot(xs, hit_prob, '-*g', label='hit')
    ax.plot(xs, miss_prob, '-*b', label='miss')
    ax.plot(xs, fa_prob, '-*r', label='fa')
    quickleg(ax, bbox_to_anchor=(-0.5,0))
    ax.set_xlabel('food reward')
    ax.set_ylabel('probability')
    # ax.set_xticks(xs, [f'{a:.0f}' for a in att_cost_list]) # actual values
    ax.set_xticks([xs[i] for i in [0,-1]], ['low', 'high']) # low high
    ax.set_xticks(xs)
    ax.set_yticks([0,0.5,1], ['0', '.5','1'])
    ax.set_xticks(xs)
    centerax(ax)
    ax.set_title('hit, miss, false alarm')
    plt.show()

    # reaction time
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    data = []
    for food_reward_idx, v in enumerate(rt_hit_dist):
        v = np.array(v)
        v = v[v > 1]  # ignore the 0 reaction time in the plots
        if len(v) == 0:
            continue
        data.append((food_reward_idx, v))


    for i, (food_reward_idx, curve) in enumerate(data):
        sns.kdeplot(curve, label=f'reward:{att_cost_list[food_reward_idx]:.0f}', clip=(1, np.max(v)), bw_adjust=1,color=rewardcmap(np.linspace(0, 1, len(data)))[i])

        # plt.plot(curve, color=cmap(np.linspace(0, 1, len(data)))[i])
    # color_bar=plt.colorbar(plt.cm.ScalarMappable(cmap=rewardcmap),ax=ax)
    # ticks = [0, 0.5, 1]
    # tick_labels = ['Low food reward', '', 'High food reward']
    # color_bar.set_ticks(ticks)
    # color_bar.set_ticklabels(tick_labels)

    ax.set_ylabel('probability')
    ax.set_xlabel('reaction time')
    centerax(ax)
    ax.set_xticks([0, 10, 20],[0, 10, 20])
    ax.set_yticks([])
    ax.set_yticklabels([])
    # quickleg(ax, bbox_to_anchor=(-0.5, 0))
    ax.set_title('vary reward \n reaction time distribution')
    # quicksave('reaction time distribution v3 full', modelname)
    plt.show()


    # total attention time distribution, figure 4C
    fig= plt.figure()
    for food_reward_idx in np.arange(len(att_cost_list)):
        
        subdf=df[(df.att_idx==food_reward_idx)]
        sns.kdeplot([len(a) for a in subdf.at_times],  label=f'food reward={att_cost_list[food_reward_idx]:.0f}', bw_adjust=2, common_norm=False, color=rewardcmap(
            np.linspace(0, 1, len(att_cost_list)))[food_reward_idx])

        plt.xlim(0, 15)
        plt.xticks([0, 15], [0,15])
        plt.xlabel('# high attentions')
        plt.ylabel('prob')
        plt.title(f'total attention times, varying food reward')
        centerax(plt.gca())
        plt.yticks([])

    plt.show()

    # fig4g
    fig, ax1 = plt.subplots(1,1, figsize=(5,4))
    for food_reward_idx in np.arange(len(att_cost_list)):
        subdf=df[df.att_idx==food_reward_idx]
        att_seq_list = []
    
        for episode_actions in subdf.actions:
            att_seq_list.append([float(a[1]) for a in episode_actions])
        
        # Sorting based on increasing trial length
        sorted_list_of_lists = sorted(att_seq_list, key=len)
        # Choosing what range of trial lengths to use. Could modify to be more consistent.
        sequences = sorted_list_of_lists[-plot_no_episodes:]
        # autocorr
        autocorr, total_counts = empirical_autocorrelation_new(
            sequences, min_no_samples=1111)
        xs = np.arange(0, len(autocorr), 1)
        xs = xs-len(xs)//2
        ax1.plot(xs, autocorr,
                    label=f'node:{1}', color=rewardcmap(np.linspace(0, 1, len(att_cost_list)))[food_reward_idx])
    

        # ax1.set_xlabel(r'$\tau$ [s]')
        # # ax1.set_title('auto correlation')
        ax1.set_xlim(-50, 50)
        # ax1.set_ylim(None, 1)
        ax1.set_xticks([-50, 0, 50])
        # ax1.set_yscale('log')
        ax1.set_yticks([1])
        ax1.spines['left'].set_position('zero')
        ax1.spines['bottom'].set_position('zero')
        ax1.tick_params(axis='x', pad=50) 

    
    fig.suptitle(f'vary nodes, peroid')
    plt.tight_layout()
    # quicksave('vary node p autocorr', 'plot vary node')
    plt.show()


In [9]:
for iepoch in range(0, n_epoch):
    # training ----------------
    for node in node_list:
        for p1p2 in plist:
            thismodel = f'seed{seed}_{modelname}_n{node}_p{p1p2}_ep{iepoch}'
        
            task = AFtask(spec={'agent':
                            {'lick_cost': env_param[0],
                            'food_reward': env_param[1],
                            'attention_cost_coeff': env_param[2],
                            'attention_cost_temp': env_param[3],
                            'penalty_cost': env_param[4],
                            'iti_cost': env_param[5],
                            'time_in_game_reward': env_param[6]},
                            'experiment': {'no_signal_nodes': node}})
            task.att_cost_list = att_cost_list
            task.food_reward=500
            task.food_reward_list=np.linspace(30,500,3)
            task.obs_certainity_possible = p1p2
            taskbelief = FuncBeliefModel(env=task, rng=1)


            # train
            if iepoch == 0:
                model = PPO('MlpPolicy', taskbelief, verbose=0, device='cpu',
                            clip_range=0.1, ent_coef=0.01)
            else:
                previous_model = f'seed{seed}_{modelname}_n{node}_p{p1p2}_ep{iepoch-1}'
                model = PPO.load(f'ycstore/{previous_model}', env=taskbelief)
            model.learn(total_timesteps=epoch_size,)
            model.save(f'ycstore/{thismodel}')
            print(f'epoch{iepoch}, node{node}: yctore/{thismodel} saved')
            model = PPO.load(f'ycstore/{thismodel}', env=taskbelief) # test load


/Users/yc/miniconda3/envs/lab/lib/python3.11/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/Users/yc/miniconda3/envs/lab/lib/python3.11/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


epoch0, node25: yctore/seed0_test_n25_p[0.5 0.7]_ep0 saved
epoch1, node25: yctore/seed0_test_n25_p[0.5 0.7]_ep1 saved
epoch2, node25: yctore/seed0_test_n25_p[0.5 0.7]_ep2 saved


In [7]:
def run_one_episode_cont(task, taskbelief, agent,
                    num_steps=1000, deterministic=True):
    '''modified run one ep function.'''
    q_states = [[i] for i in range(task.no_nodes)]

    actions, rewards, states, observations, beliefs = [], [], [], [], []
    trial_food_reward = []

    def get_queries(env): return q_states
    try:
        _q_states = get_queries(taskbelief.env)
        _q_states, _q_probs = [], []
    except:
        get_queries = None
    task.reset()
    p1p2 = task.obs_certainity_possible
    belief, info = taskbelief.reset(task, return_info=True)
    states.append(info['state'])
    observations.append(info['observation'])
    beliefs.append(belief)
    if get_queries is not None:
        _q_states.append(np.array(get_queries(taskbelief.env)))
        _q_probs.append(taskbelief.query_probs(_q_states[-1]))
    t = 0
    while True:

        action, _ = agent.predict(belief, deterministic=deterministic)
        # action, _ = agent.predict(belief)
        action = action

        actions.append(action)
        belief, reward, done, info = taskbelief.step(action, task)
        rewards.append(reward)
        states.append(info['state'])
        observations.append(info['observation'])
        beliefs.append(belief)
        if get_queries is not None:
            _q_states.append(np.array(get_queries(taskbelief.env)))
            _q_probs.append(taskbelief.query_probs(_q_states[-1]))
        t += 1
        if done or t == num_steps:
            break
    # print(observations)
    episode = {
        'trial_food_reward_idx': task.food_reward_idx,
        'num_steps': t,
        'actions': np.array(actions),  # [0, t)
        'rewards': np.array(rewards),  # [0, t)
        'states': np.array(states),  # [0, t]
        'observations': np.array(observations),  # [0, t]
        'beliefs': np.array(beliefs),  # [0, t]
        'p1p2': p1p2
    }

    if get_queries is not None:
        diffs = ((_q_states-_q_states[0]) **
                 2).reshape(len(_q_states), -1).sum(axis=1)
        if np.all(diffs < 1e-8):  # merge fixed query set
            _q_states = _q_states[0]
        # (num_queries, state_dim, t+1) or (num_queries, state_dim)
        episode['q_states'] = np.array(_q_states)
        episode['q_probs'] = np.array(_q_probs)  # (num_queries, t+1)

    return episode


In [8]:
def eval_wrapper(a):
    episode = run_one_episode_cont(task=task, taskbelief=taskbelief, agent=model,
                            num_steps=100000, deterministic=True)
    return episode


results=[]
with multiprocess.Pool(processes=8) as pool:
    all_episode_data = pool.map(eval_wrapper, range(no_episodes))

for episode in all_episode_data:
    # process
    # episode['inode'] = inode
    # episode['ip'] = ip
    episode['at_times'] = np.where(
        np.array([elt[1] for elt in episode['actions']]) == 1)[0].astype('int')
    episode['at_seq'] = [float(elt[1])
                        for elt in episode['actions']]
    if episode['actions'][-1][0] == 1:
        episode['licktime'] = int(len(episode['states']))
    else:
        episode['lick_licktimetime'] = -1
    episode['trial_len'] = len(episode['actions'])
    
    results.append(episode)

df = pd.DataFrame(results)
print(f'total reward: {sum(np.concatenate(df.rewards.to_numpy()))}')
hit_reaction_plot(df)

AttributeError: 'AuditoryForaging' object has no attribute 'food_reward_idx'

In [ ]:

    # # eval collection --------------------
    # results=[]
    # for inode, node in enumerate(node_list):
    #     for ip, p1p2 in enumerate(plist):
    #         task = AFtask(spec={'agent': 
    #                                     {'lick_cost': env_param[0],
    #                                     'food_reward': env_param[1],
    #                                     'attention_cost_coeff': env_param[2],
    #                                     'attention_cost_temp': env_param[3],
    #                                     'penalty_cost': env_param[4],
    #                                     'iti_cost': env_param[5],
    #                                     'time_in_game_reward': env_param[6]},
    #                                     'experiment':{                              'no_signal_nodes':node}})
    #         task.att_cost_list = att_cost_list
    #         taskbelief = FuncBeliefModel(env=task, rng=1)
    #         task.obs_certainity_possible = p1p2
    #         thismodel = f'seed{seed}_{modelname}_n{node}_p{p1p2}_ep{iepoch}'
    #         model = PPO.load(f'ycstore/{thismodel}')

    #         def eval_wrapper(a):
    #             episode = run_one_episode_attcost(task=task, taskbelief=taskbelief, agent=model,
    #                                     num_steps=100000, deterministic=True)
    #             return episode

    #         with multiprocess.Pool(processes=8) as pool:
    #             all_episode_data = pool.map(eval_wrapper, range(no_episodes))
            
    #         for episode in all_episode_data:
    #             # process
    #             episode['inode'] = inode
    #             episode['ip'] = ip
    #             episode['at_times'] = np.where(
    #                 np.array([elt[1] for elt in episode['actions']]) == 1)[0].astype('int')
    #             episode['at_seq'] = [float(elt[1])
    #                                 for elt in episode['actions']]
    #             if episode['actions'][-1][0] == 1:
    #                 episode['licktime'] = int(len(episode['states']))
    #             else:
    #                 episode['lick_licktimetime'] = -1
    #             episode['trial_len'] = len(episode['actions'])
                
    #             results.append(episode)

    # df = pd.DataFrame(results)
    # print(f'total reward: {sum(np.concatenate(df.rewards.to_numpy()))}')
    # hit_reaction_plot(df)

